In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("amazon_uk_products.csv")

try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
except Exception as e:
    print("Default CSV read failed; trying engine='python'...\n", e)
    df = pd.read_csv(DATA_PATH, low_memory=False, engine='python')

print("Rows:", len(df), "Columns:", len(df.columns))
display(df.head(3))
list(df.columns)[:50] 


In [ ]:

def _norm(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")

def guess_column(df, candidates):
    cols = { _norm(c): c for c in df.columns }
    for cand in candidates:
        key = _norm(cand)
        
        if key in cols:
            return cols[key]
      
        for k, orig in cols.items():
            if key in k:
                return orig
    return None

cand_category = ["category", "primarycategory", "main_category", "product_category"]
cand_brand    = ["brand", "brands"]
cand_price    = ["price", "current_price", "sale_price", "offer_price", "discounted_price", "listing_price", "amount"]
cand_stars    = ["stars", "rating", "review_rating", "star_rating", "avg_rating"]
cand_best     = ["isBestSeller", "is_bestseller", "bestseller", "is_best_seller", "best_seller"]

col_category = guess_column(df, cand_category)
col_brand    = guess_column(df, cand_brand)
col_price    = guess_column(df, cand_price)
col_stars    = guess_column(df, cand_stars)
col_best     = guess_column(df, cand_best)

print("Auto-detected:")
print("  category  ->", col_category)
print("  brand     ->", col_brand)
print("  price     ->", col_price)
print("  stars     ->", col_stars)
print("  isBestSel ->", col_best)

if col_category is None or col_price is None:
    raise ValueError("Please set 'col_category' and 'col_price' to valid columns and re-run this cell.")


In [ ]:

import re

def to_numeric_price(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int,float)): return float(v)
    s = str(v).replace(",", ".")
    s = re.sub(r"[^0-9.]", "", s)
    try:
        return float(s) if s else np.nan
    except:
        return np.nan

df1 = df.copy()

# price
df1["_price"] = df1[col_price].apply(to_numeric_price)
df1.loc[df1["_price"] < 0, "_price"] = np.nan

# stars
if col_stars and col_stars in df1.columns:
    df1["_stars"] = pd.to_numeric(df1[col_stars], errors="coerce")
    df1.loc[(df1["_stars"] < 0) | (df1["_stars"] > 5), "_stars"] = np.nan
else:
    df1["_stars"] = np.nan

# best seller
def to_boolish(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if s in {"true","yes","y","1"}: return 1
    if s in {"false","no","n","0"}: return 0
    # if it's already numeric-like:
    try:
        f = float(s)
        return 1 if f != 0 else 0
    except:
        return np.nan

if col_best and col_best in df1.columns:
    df1["_best"] = df1[col_best].apply(to_boolish)
else:
    
    df1["_best"] = 0.0


df1["_category"] = df1[col_category].astype(str).str.strip()
df1["_brand"]    = df1[col_brand].astype(str).str.strip() if col_brand else ""

df1[["_category","_brand","_price","_stars","_best"]].head(5)


In [ ]:

# Crosstab
ct = pd.crosstab(df1["_category"], df1["_best"]).rename(columns={0:"not_best", 1:"best"})
display(ct.head(10))

# Proportion of best-sellers x category
prop_best = (ct.div(ct.sum(axis=1), axis=0).fillna(0).sort_values("best", ascending=False))
display(prop_best.head(10))

ct.to_csv("crosstab_category_best.csv")
prop_best.to_csv("proportion_best_by_category.csv")
print("Saved: crosstab_category_best.csv, proportion_best_by_category.csv")


In [ ]:

# Chi-square test of independence
try:
    from scipy.stats import chi2_contingency
    use_scipy = True
except Exception as e:
    print("SciPy not found; chi-square will be computed via numpy approximation.")
    use_scipy = False

if use_scipy:
    chi2, p, dof, expected = chi2_contingency(ct.fillna(0))
else:
    # Fallback using numpy (G-test approximation not included here); recommend SciPy for exact
    obs = ct.fillna(0).values
    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    expected = row_sums @ col_sums / total
    # chi2 = sum((O-E)^2/E)
    chi2 = np.nansum((obs - expected)**2 / np.where(expected==0, np.nan, expected))
    dof = (obs.shape[0]-1)*(obs.shape[1]-1)
    # p-value not computed without SciPy; set to NaN
    p = np.nan

print("Chi-square:", chi2, "  dof:", dof, "  p-value:", p)

# Cramér's V
n = ct.values.sum()
k = min(ct.shape)-1
if n > 0 and k > 0:
    cramers_v = np.sqrt((chi2 / n) / k)
else:
    cramers_v = np.nan
print("Cramér's V:", round(cramers_v, 4))


In [ ]:

# stacked bars
top20 = ct.sum(axis=1).sort_values(ascending=False).head(20).index
ct_top = ct.loc[top20]

import matplotlib.pyplot as plt

plt.figure()
x = np.arange(len(ct_top))
best = ct_top["best"] if "best" in ct_top.columns else np.zeros(len(ct_top))
notbest = ct_top["not_best"] if "not_best" in ct_top.columns else np.zeros(len(ct_top))

plt.bar(x, notbest, label="Not best")
plt.bar(x, best, bottom=notbest, label="Best")
plt.xticks(x, ct_top.index, rotation=45, ha="right")
plt.title("Category vs Best-seller (stacked counts — top 20 categories)")
plt.xlabel("Category")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

price = df1["_price"].dropna()
q1, q3 = price.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr

df_no_ol = df1[(df1["_price"].notna()) & (df1["_price"] >= low) & (df1["_price"] <= high)].copy()
print("Kept rows:", len(df_no_ol), " / ", len(df1))

df_no_ol[["_category","_price","_stars"]].head(3)


In [ ]:

# top categories x count
top20_cats = (df_no_ol["_category"].value_counts().head(20).index.tolist())
data_list = [df_no_ol.loc[df_no_ol["_category"]==c, "_price"].values for c in top20_cats]

plt.figure()
plt.violinplot(data_list, showmedians=True)
plt.xticks(range(1, len(top20_cats)+1), top20_cats, rotation=45, ha="right")
plt.title("Price distribution by Category (violin, top 20; outliers removed)")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


In [ ]:

med_by_cat_full = (df1.dropna(subset=["_price"])
                     .groupby("_category")["_price"]
                     .median()
                     .sort_values(ascending=False))
display(med_by_cat_full.head(10))
print("Highest median price category:", med_by_cat_full.index[0], "→", round(med_by_cat_full.iloc[0],2))


In [ ]:

top10_cats = df_no_ol["_category"].value_counts().head(10).index
avg_price_top10 = (df_no_ol[df_no_ol["_category"].isin(top10_cats)]
                   .groupby("_category")["_price"]
                   .mean()
                   .sort_values(ascending=False))

plt.figure()
plt.bar(avg_price_top10.index, avg_price_top10.values)
plt.xticks(rotation=45, ha="right")
plt.title("Average price by category (top 10 by count; outliers removed)")
plt.xlabel("Category")
plt.ylabel("Average price")
plt.tight_layout()
plt.show()


In [ ]:

mean_by_cat_full = (df1.dropna(subset=["_price"])
                     .groupby("_category")["_price"]
                     .mean()
                     .sort_values(ascending=False))
display(mean_by_cat_full.head(10))
print("Highest average price category:", mean_by_cat_full.index[0], "→", round(mean_by_cat_full.iloc[0],2))


In [ ]:
if df_no_ol["_stars"].notna().sum() == 0:
    print("No rating data — skipping box plots.")
else:
    top10_cats_r = df_no_ol["_category"].value_counts().head(10).index.tolist()
    data_list_r = [df_no_ol.loc[(df_no_ol["_category"]==c) & (df_no_ol["_stars"].notna()), "_stars"].values
                   for c in top10_cats_r]

    import matplotlib.pyplot as plt
    plt.figure()
    plt.boxplot(data_list_r, showmeans=True)
    plt.xticks(range(1, len(top10_cats_r)+1), top10_cats_r, rotation=45, ha="right")
    plt.title("Ratings by Category (box plots; top 10; outliers removed in price)")
    plt.ylabel("Stars")
    plt.tight_layout()
    plt.show()


In [ ]:

if df1["_stars"].notna().sum() == 0:
    print("No rating data available.")
else:
    med_rating_full = (df1.dropna(subset=["_stars"])
                         .groupby("_category")["_stars"]
                         .median()
                         .sort_values(ascending=False))
    display(med_rating_full.head(10))
    print("Highest median rating category:", med_rating_full.index[0], "→", round(med_rating_full.iloc[0],2))


In [ ]:

if df1[["_price","_stars"]].dropna().empty:
    print("Not enough data for correlation.")
else:
    corr_pearson  = df1[["_price","_stars"]].corr(method="pearson").iloc[0,1]
    corr_spearman = df1[["_price","_stars"]].corr(method="spearman").iloc[0,1]
    print("Pearson r(price, stars):", round(corr_pearson, 4))
    print("Spearman ρ(price, stars):", round(corr_spearman, 4))


In [ ]:

if df_no_ol[["_price","_stars"]].dropna().empty:
    print("Not enough data to plot scatter.")
else:
    import matplotlib.pyplot as plt
    d2 = df_no_ol[["_price","_stars"]].dropna()
    plt.figure()
    plt.scatter(d2["_stars"], d2["_price"], s=8)  # small markers
    plt.title("Price vs Rating (stars) — outliers removed in price")
    plt.xlabel("Stars")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()


In [ ]:

num_cols = df1.select_dtypes(include=[np.number]).columns
if len(num_cols) == 0:
    print("No numeric columns for heatmap.")
else:
    import matplotlib.pyplot as plt
    corr = df1[num_cols].corr()
    plt.figure()
    plt.imshow(corr, aspect="auto")
    plt.title("Correlation heatmap (numeric columns)")
    plt.colorbar()
    plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
    plt.yticks(range(len(num_cols)), num_cols)
    plt.tight_layout()
    plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

price_clean = df_no_ol["_price"].dropna().values
if price_clean.size < 10:
    print("Not enough data for QQ plot.")
else:
    try:
        from scipy import stats
        plt.figure()
        stats.probplot(price_clean, dist="norm", plot=plt)
        plt.title("QQ plot — price vs normal (outliers removed)")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        try:
            price_sorted = np.sort(price_clean)
            n = price_sorted.size
            probs = (np.arange(1, n+1) - 0.5) / n
            
            if hasattr(np, "erfinv"):
                theo = np.sqrt(2) * np.erfinv(2*probs - 1)
                plt.figure()
                plt.scatter(theo, price_sorted, s=8)
                # add reference line
                coef = np.polyfit(theo, price_sorted, 1)
                fit = np.poly1d(coef)
                xline = np.linspace(theo.min(), theo.max(), 100)
                plt.plot(xline, fit(xline))
                plt.title("QQ plot (fallback) — price vs normal")
                plt.xlabel("Theoretical quantiles")
                plt.ylabel("Observed price")
                plt.tight_layout()
                plt.show()
            else:
                print("QQ plot fallback requires numpy.erfinv (not found). Install SciPy for probplot.")
        except Exception as e2:
            print("Could not draw QQ plot:", e2)


In [ ]:

# Violin
top20_cats_u = df1["_category"].value_counts().head(20).index.tolist()
data_list_u = [df1.loc[df1["_category"]==c, "_price"].dropna().values for c in top20_cats_u]

plt.figure()
plt.violinplot(data_list_u, showmedians=True)
plt.xticks(range(1, len(top20_cats_u)+1), top20_cats_u, rotation=45, ha="right")
plt.title("Price by Category (violin, top 20; **no** outlier removal)")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


In [ ]:

lines = []

# Part 1 highlights
if 'prop_best' in locals() and not prop_best.empty:
    top_best = prop_best['best'].head(5)
    lines.append("Categories with highest share of best-sellers (top 5): " +
                 ", ".join(f"{idx} ({val:.2%})" for idx, val in top_best.items()))

# Part 2 highlights
if 'med_by_cat_full' in locals() and len(med_by_cat_full)>0:
    lines.append(f"Highest median price category: {med_by_cat_full.index[0]} (~£{med_by_cat_full.iloc[0]:.2f}).")
if 'mean_by_cat_full' in locals() and len(mean_by_cat_full)>0:
    lines.append(f"Highest average price category: {mean_by_cat_full.index[0]} (~£{mean_by_cat_full.iloc[0]:.2f}).")
if 'med_rating_full' in locals() and len(med_rating_full)>0:
    lines.append(f"Highest median rating category: {med_rating_full.index[0]} ({med_rating_full.iloc[0]:.2f} stars).")

# Part 3 highlights
try:
    lines.append(f"Price ↔ stars: Pearson {corr_pearson:.3f}, Spearman {corr_spearman:.3f}.")
except:
    lines.append("Price ↔ stars: insufficient data for correlation.")

report = "\n- ".join(["Business Summary:"] + lines) if lines else "Business Summary: (insufficient data — please check column mapping and rerun)"
print(report)

with open("bivariate_business_summary.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("\nSaved: bivariate_business_summary.txt")
